In [7]:
!pip install simpy

In [8]:
import random
import statistics

import simpy

class Response(object):
  def __init__(self, label):
    self.label = label
    self.sum = 0

  def add(self, x):
    self.sum += x

  def value(self):
    return self.sum

  def reset(self):
    self.sum = 0

class DiscreteResponse(Response):
  def __init__(self, label):
    super().__init__(label)
    self.number = 0

  def add(self, x):
    super().add(x)
    self.number += 1

  def value(self):
    return super().value() / self.number

  def reset(self):
    super().reset()
    self.number = 0

class Store(object):

    time_of_run = Response("time of run")
    number_of_buyers = Response("number of buyers")
    time_in_store = DiscreteResponse("time in store")

    def __init__(self, env, num_cash_terminals):
        self.env = env
        self.cash_terminal = simpy.Resource(env, num_cash_terminals)


class Buyer(object):
    number = 0

    def __init__(self, number, arrival_time):
      self.number = number
      self.arrival_time = arrival_time

    def go_to_store(self, env, store):
        saleroom_time = random.uniform(MEAN_TIME_IN_SALEROOM - DELTA, MEAN_TIME_IN_SALEROOM + DELTA)
        yield env.timeout(saleroom_time)
        log('fs', env, self)

        with store.cash_terminal.request() as request:
            yield request
            log('sp', env, self)
            yield env.timeout(max(0.5, random.normalvariate(saleroom_time * K_SERVICE_TIME, S_SERVICE_TIME)))
            log('fp', env, self)

            Store.number_of_buyers.add(1)
            Store.time_in_store.add(env.now - self.arrival_time)

def run_store(env, run_time, num_cash_terminals, frequency, num_buyers_at_start):

    def start_buyer():
        buyer = Buyer(Buyer.number, env.now)
        Buyer.number += 1
        log('ba', env, buyer)
        env.process(buyer.go_to_store(env, store))

    store = Store(env, num_cash_terminals)

    # warm up
    for _ in range(num_buyers_at_start):
        start_buyer()

    # start
    while env.now < run_time:
        yield env.timeout(random.expovariate(frequency))
        if env.now < run_time:
          start_buyer()



def log(act, env, buyer):
    if(IS_LOGGED):
      print(f"{act} {env.now:7.2f} {buyer.number:3d}")


def main(run_time, num_cash_terminals, frequency, num_buyers_at_start):
    # Setup
    # Run the simulation
    Store.time_of_run.reset();
    Store.number_of_buyers.reset();
    Store.time_in_store.reset();

    env = simpy.Environment()
    env.process(
        run_store(env, run_time, num_cash_terminals, frequency, num_buyers_at_start)
    )
    env.run(None) # service with clearance
    Store.time_of_run.add(env.now)

    # View the results
    print(
        f"{Store.time_of_run.label} = {Store.time_of_run.value():.2f} minutes",
    )
    print(
        f"{Store.number_of_buyers.label} = {Store.number_of_buyers.value()}",
    )
    print(
        f"{Store.time_in_store.label} = {Store.time_in_store.value():.2f} minutes",
    )
    assert Store.time_of_run.value() >= run_time
    print("---")


if __name__ == "__main__":
  random.seed(45)
  NUM_REPLICATIONS = 10
  IS_LOGGED = NUM_REPLICATIONS == 1
  RUN_TIME = 60
  NUM_BUYERS_AT_START = 5
  NUM_CASH_TERMINALS = 4
  FREQUENCY = 3
  MEAN_TIME_IN_SALEROOM = 4
  DELTA = 2
  K_SERVICE_TIME = 0.3
  S_SERVICE_TIME = 0.5
  for _ in range(NUM_REPLICATIONS):
    main(RUN_TIME, NUM_CASH_TERMINALS, FREQUENCY, NUM_BUYERS_AT_START)

time of run = 65.56 minutes
number of buyers = 161
time in store = 5.52 minutes
---
time of run = 68.73 minutes
number of buyers = 210
time in store = 8.24 minutes
---
time of run = 66.16 minutes
number of buyers = 175
time in store = 6.28 minutes
---
time of run = 66.20 minutes
number of buyers = 170
time in store = 5.56 minutes
---
time of run = 67.81 minutes
number of buyers = 177
time in store = 6.00 minutes
---
time of run = 66.42 minutes
number of buyers = 168
time in store = 6.11 minutes
---
time of run = 66.80 minutes
number of buyers = 173
time in store = 6.00 minutes
---
time of run = 67.11 minutes
number of buyers = 153
time in store = 5.58 minutes
---
time of run = 65.54 minutes
number of buyers = 190
time in store = 6.25 minutes
---
time of run = 68.57 minutes
number of buyers = 206
time in store = 7.67 minutes
---
